# ORIE 5355 Project 2026: Part 2 head-to-head testing


### Part 2: competition with capacity constraints

Run `model_training/purchase_model_baseline.ipynb` first; it saves the model loaded by `agents/agent_part2.py`. This notebook tests that demand-based starter agent against another agent. The starter uses a simple inventory heuristic but does not model its opponent; improve this baseline for competition. Inventory is a uniformly random integer from 7 to 20 every 20 customers. Use `run_project_part1_static_2026.ipynb` for unconstrained static pricing and `run_project_part1_capacity_2026.ipynb` for capacity-constrained pricing without competition.

The two Part 2 example opponents are `agents/part2_dummy_fixed_prices.py` and `agents/part2_dummy_fixed_prices_adaptive.py`. Try your agent against itself and these dummy agents by changing `agentnames` below. Compare average revenue per customer over several fresh trials; the default 500 customers is a quick test, and you can increase `T` to 2500 for longer tests.

For many trials, you may find it easier to convert this notebook into a Python script, turn off plotting, and collect each trial's revenue. Start each trial with a fresh environment and agents.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / 'project_code_2026_publicshare',
                   parent / 'project_2026/project_code_2026_publicshare',
                   parent / 'private_github/project_2026/project_code_2026_publicshare')
    if (folder / 'pricing_helpers.py').is_file()
), None)
if project_folder is None:
    raise FileNotFoundError('Open the project folder as your notebook workspace.')
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

import copy
import numpy as np 
import matplotlib.pyplot as plt
import time
from IPython import display

In [ ]:
import make_env_2026 as make_env

### Part 2: head-to-head simulation


In [ ]:
# In part 2, each team's agent will be evaluated against all other teams.
project_part = 2
agentnames = ['agent_part2', 'part2_dummy_fixed_prices_adaptive']

In [ ]:
env, agents = make_env.make_env_agents(
    agentnames = agentnames, 
    project_part = project_part, 
    first_file = 'data/datafile1_2026.csv', 
    second_file='data/datafile2_2026.csv'
    )

In [ ]:
T = 500 #2500
verbose = False

In [ ]:
# The starter agent uses its saved demand model but does not account for its opponent.

env.reset()
customer_covariates, sale, profits, inventories, time_until_replenish = env.get_current_state_customer_to_send_agents()
last_customer_covariates = customer_covariates
inventory_history = [[], []]

fig, ax = plt.subplots(figsize=(20, 10))
for t in range(0, T):
    actions = [agent.action(copy.deepcopy((customer_covariates, sale, profits, inventories, time_until_replenish))) for agent in agents]
    customer_covariates, sale, profits, inventories, time_until_replenish = env.step(actions)
    inventory_history[0].append(inventories[0])
    inventory_history[1].append(inventories[1])
    newplot = env.render(True)
    if newplot:
        display.clear_output(wait=True)
        display.display(plt.gcf())
    if verbose:
        print('last customer covariate: ', last_customer_covariates)
        print('last (item bought, agent bought from, prices): ', sale)
        print('current_profit per agent: ', profits)
    last_customer_covariates = customer_covariates

plt.close()
print("Cumulative profit: {}".format(env.agent_profits))
print("Average revenue per customer: {}".format(np.asarray(env.agent_profits)/T))
print("Cumulative buyer utility: {}".format(env.cumulative_buyer_utility))

In [ ]:
plt.plot(inventory_history[0], label = agentnames[0])
plt.plot(inventory_history[1], label = agentnames[1])
plt.legend()